# calibra — quickstart

This notebook walks through the recommended usage of **calibra**: uncertainty quantification for language model outputs.

It covers:
1. Installation
2. Load a model
3. Single-prompt uncertainty estimation
4. Benchmarking on a custom dataset
5. Metrics and visualisation

## 1. Installation

In [ ]:
%pip install git+https://github.com/tjoliveira/calibra.git

In [ ]:
import calibra
print(calibra.__version__)

## 2. Load a model

`Estimator.from_pretrained` accepts any HuggingFace causal LM.  
Use a small model here so the notebook runs on CPU in a reasonable time.  
On a GPU machine, swap in `Qwen/Qwen3-8B` (or larger) for better results.

In [ ]:
from calibra import Estimator

# Small model — fast enough for a notebook demo
# Replace with e.g. "Qwen/Qwen3-8B" for production use
MODEL = "Qwen/Qwen3-0.6B"

est = Estimator.from_pretrained(
    MODEL,
    trust_remote_code=True,                 # required for some Qwen checkpoints
    semantic_model="all-MiniLM-L6-v2",      # needed for self_consistency / bsdetector
)

## 3. Single-prompt uncertainty estimation

Call `est.estimate()` with any text prompt.  
Scores closer to **0 = confident**, higher = more uncertain.

**Probability-based** methods (`entropy`, `max_probability`, `sequence_probability`) need one forward pass.  
**Sampling-based** methods (`self_consistency`, `self_reflection`, `bsdetector`) need additional passes.

In [ ]:
# Probability-based methods: one forward pass, very fast
scores = est.estimate(
    "What is the capital of France?",
    methods=["entropy", "max_probability", "sequence_probability"],
)
print(scores)

In [ ]:
# Sampling-based methods: multiple forward passes + semantic similarity
scores_all = est.estimate(
    "What is the capital of France?",
    methods=["entropy", "self_consistency", "self_reflection", "bsdetector"],
    num_samples=3,      # keep low for the demo
    max_new_tokens=50,
)
print(scores_all)

In [ ]:
# Compare a factual question vs. a speculative one
factual     = est.estimate("What is 2 + 2?",                   methods=["entropy"])
speculative = est.estimate("What will happen to AI in 2050?",  methods=["entropy"])

print(f"Factual    entropy: {factual['entropy']:.4f}")
print(f"Speculative entropy: {speculative['entropy']:.4f}")

### Token-level breakdown

`score_all()` returns token-level uncertainty for the probability-based methods.

In [ ]:
detailed = est.score_all(
    "What is the tallest mountain on Earth?",
    max_new_tokens=30,
)

print("sentence entropy :", detailed["entropy"]["sentence"])
print("per-token entropy:", [f"{x:.3f}" for x in detailed["entropy"]["tokens"]])

## 4. Benchmark on a custom dataset

Pass a list of `{"input": ..., "target": ...}` dicts to `bench.run()`.  
Results expose:
- `results.uncertainties` — per-method, per-sample scores  
- `results.errors` — per-sample error values  
- `results.auroc` / `results.ece` — aggregated metrics

In [ ]:
from calibra import Benchmark

custom_data = [
    {"input": "Who wrote Hamlet?",                       "target": "Shakespeare"},
    {"input": "What year did World War II end?",         "target": "1945"},
    {"input": "What is the chemical formula of water?",  "target": "H2O"},
    {"input": "Who painted the Mona Lisa?",              "target": "Leonardo da Vinci"},
    {"input": "What is the speed of light in m/s?",      "target": "299792458"},
]

bench = Benchmark(est)
results = bench.run(
    task="qa",
    dataset=custom_data,
    methods=["entropy", "max_probability", "sequence_probability"],
    max_new_tokens=10,
)

print("AUROC:", results.auroc)
print("ECE  :", results.ece)

In [ ]:
# Inspect per-sample scores
import pandas as pd

df = pd.DataFrame({
    "question": [d["input"] for d in custom_data],
    "error"   : results.errors,
    **{m: results.uncertainties[m] for m in results.uncertainties},
})
df

## 5. Metrics

The `calibra.metrics` module exposes each metric individually.

In [ ]:
from calibra import metrics

u = results.uncertainties["entropy"]
e = results.errors

print("AUROC            :", metrics.auroc(u, e))
ece_score, _ = metrics.ece(u, e)
print("ECE              :", ece_score)
print("Correlations     :", metrics.correlations(u, e))
print("Uncertainty stats:", metrics.uncertainty_stats(u))

In [ ]:
# All metrics in one call
summary = metrics.summarize(u, e)
print(summary)

## 6. Visualisation

`calibra.viz` provides ready-made plots that also accept a Matplotlib `ax=` argument for composability.

In [ ]:
from calibra import viz
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

viz.roc_curve(
    results.uncertainties["entropy"],
    results.errors,
    method_name="entropy",
    ax=axes[0],
)

viz.calibration_curve(
    results.uncertainties["entropy"],
    results.errors,
    method_name="entropy",
    ax=axes[1],
)

viz.uncertainty_distribution(
    results.uncertainties,
    ax=axes[2],
)

plt.tight_layout()
plt.show()

from calibra import viz
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

viz.roc_curve(
    results.uncertainties["entropy"],
    results.errors,
    method_name="entropy",
    ax=axes[0],
)

viz.calibration_curve(
    results.uncertainties["entropy"],
    results.errors,
    method_name="entropy",
    ax=axes[1],
)

viz.distribution(
    results.uncertainties["entropy"],
    method_name="entropy",
    ax=axes[2],
)

plt.tight_layout()
plt.show()

In [ ]:
# Uncomment to run — takes a few minutes depending on hardware

# results_squad = bench.run(
#     task="qa",
#     dataset="squad",
#     methods=["entropy", "max_probability"],
#     max_samples=50,
# )
# print(results_squad.auroc)